In [2]:
import json
import os
import random
from collections import defaultdict
from typing import List, Dict, Set
NETSCRATCH_DIR = '/netscratch/elmansoury'
MSCOCO_DIR = f'{NETSCRATCH_DIR}/mscoco/train2014/'

# print the difference in the paths
print(os.path.join(MSCOCO_DIR,'annotations/annotations/instances_train2014.json'))
print(f'{MSCOCO_DIR}/annotations/annotations/instances_train2014.json')

with open(os.path.join(MSCOCO_DIR,'annotations/annotations/instances_train2014.json'), 'r') as f:
    coco_data = json.load(f)

/netscratch/elmansoury/mscoco/train2014/annotations/annotations/instances_train2014.json
/netscratch/elmansoury/mscoco/train2014//annotations/annotations/instances_train2014.json


In [9]:
coco_data['categories']
# Build dictionary mapping category IDs to names, supercategories
# Name: category name, Supercategory: category supercategory
category_id_to_name = {
    category['id']: {
        'name': category['name'],
        'supercategory': category['supercategory']
    }
    for category in coco_data['categories']
}

# save the category_id_to_name as json
with open('category_id_to_name.json', 'w') as f:
    json.dump(category_id_to_name, f)

In [1]:
import json
import os
import random
from collections import defaultdict
import shutil
from typing import List, Dict, Set, Tuple


NETSCRATCH_DIR = '/netscratch/elmansoury'
MSCOCO_DIR = f'{NETSCRATCH_DIR}/mscoco/train2014/'
# print(os.path.join(MSCOCO_DIR,'/annotations/annotations/instances_train2014.json'))
# Load the COCO annotations
# with open(f'{MSCOCO_DIR}/annotations/annotations/instances_train2014.json', 'r') as f:
#     coco_data = json.load(f)


class COCOImageFilter:
    """Filter COCO images by number of unique categories."""
    
    def __init__(self, annotation_file_path: str):
        """
        Initialize the filter with COCO annotations.
        
        Args:
            annotation_file_path: Path to COCO annotations JSON file
        """
        self.annotation_file_path = annotation_file_path
        self.coco_data = self._load_annotations()
        self.image_categories = self._build_image_category_mapping()
    
    def _load_annotations(self) -> Dict:
        """Load COCO annotations from JSON file."""
        with open(self.annotation_file_path, 'r') as f:
            return json.load(f)
    
    def _build_image_category_mapping(self) -> Dict[int, Set[int]]:
        """Build mapping of image_id to set of category_ids."""
        image_categories = defaultdict(set)
        
        for annotation in self.coco_data['annotations']:
            image_id = annotation['image_id']
            category_id = annotation['category_id']
            image_categories[image_id].add(category_id)
        
        return dict(image_categories)

    def filter_by_category_count(self, num_categories: int) -> Tuple[List[int], int]:
        """
        Filter images by exact number of unique categories.
        
        Args:
            num_categories: Exact number of unique categories required
            
        Returns:
            List of image IDs matching the criteria
        """
        filtered_images = [
            image_id for image_id, categories in self.image_categories.items()
            if len(categories) == num_categories
        ]
        unique_categories_found = set()

        # print how many images in each category
        category_count = defaultdict(int)
        for image_id in filtered_images:
            for category_id in self.image_categories[image_id]:
                category_count[category_id] += 1
                unique_categories_found.add(category_id)

        print("\nNumber of images per category in filtered results:")
        for category_id, count in category_count.items():
            print(f"  Category ID {category_id}: {count} images")

        print(f"\nTotal unique categories found in filtered images: {len(unique_categories_found)}")

        return filtered_images, len(unique_categories_found)
    
    def get_image_info(self, image_ids: List[int]) -> List[Dict]:
        """
        Get detailed information for given image IDs.
        
        Args:
            image_ids: List of image IDs
            
        Returns:
            List of dictionaries containing image information
        """
        image_info_list = []
        image_id_set = set(image_ids)
        
        for image in self.coco_data['images']:
            if image['id'] in image_id_set:
                categories = list(self.image_categories[image['id']])
                image_info_list.append({
                    'image_id': image['id'],
                    'file_name': image['file_name'],
                    'width': image.get('width'),
                    'height': image.get('height'),
                    'category_ids': categories,
                    'num_categories': len(categories)
                })
        
        return image_info_list
    
    def sample_images(self, image_ids: List[int], max_samples: int, unique_categories_found: int,
                     random_seed: int = None) -> List[int]:
        """
        Sample a maximum number of images from the filtered list.
        Sampling balanced across categories where each category is equally represented.
        
        Args:
            image_ids: List of image IDs to sample from
            max_samples: Maximum number of samples to return
            unique_categories_found: Number of unique categories found in filtered images
            random_seed: Random seed for reproducibility (optional)
            
        Returns:
            Sampled list of image IDs
        """
        if len(image_ids) <= max_samples:
            return image_ids
        if random_seed is not None:
            random.seed(random_seed)
        
        images_per_category = max_samples // unique_categories_found
        category_to_images = defaultdict(list)
        for image_id in image_ids:
            for category_id in self.image_categories[image_id]:
                category_to_images[category_id].append(image_id)

        sampled_images = []
        for category_id, images in category_to_images.items():
            if len(images) > images_per_category:
                sampled_images.extend(random.sample(images, images_per_category))
            else:
                sampled_images.extend(images)

        print(f"\nTotal balanced sampled images: {len(sampled_images)}")

        return sampled_images
    
    def save_results(self, image_info: List[Dict], output_file: str,
                    total_filtered: int, num_categories: int):
        """
        Save filtered results to JSON file.
        
        Args:
            image_info: List of image information dictionaries
            output_file: Path to output JSON file
            total_filtered: Total number of images before sampling
            num_categories: Number of categories used for filtering
        """
        output_data = {
            'filter_criteria': {
                'num_categories': num_categories,
                'total_matched': total_filtered,
                'sampled': len(image_info)
            },
            'dataset_stats': {
                'total_images': len(self.coco_data['images']),
                'total_annotations': len(self.coco_data['annotations'])
            },
            'images': image_info
        }
        
        with open(output_file, 'w') as f:
            json.dump(output_data, f, indent=2)
    
    def copy_images_to_folder(self, image_info: List[Dict], 
                             source_dir: str, dest_dir: str) -> int:
        """
        Copy filtered images to a destination folder.
        
        Args:
            image_info: List of image information dictionaries
            source_dir: Source directory containing the images
            dest_dir: Destination directory to copy images to
            
        Returns:
            Number of images successfully copied
        """
        # Create destination directory if it doesn't exist
        os.makedirs(dest_dir, exist_ok=True)
        
        copied_count = 0
        failed_count = 0
        
        print(f"\nCopying images to: {dest_dir}")
        
        for img in image_info:
            source_path = os.path.join(source_dir, img['file_name'])
            dest_path = os.path.join(dest_dir, img['file_name'])
            
            try:
                if os.path.exists(source_path):
                    shutil.copy2(source_path, dest_path)
                    copied_count += 1
                    if copied_count % 100 == 0:
                        print(f"  Copied {copied_count}/{len(image_info)} images...")
                else:
                    print(f"  Warning: Source file not found: {source_path}")
                    failed_count += 1
            except Exception as e:
                print(f"  Error copying {img['file_name']}: {e}")
                failed_count += 1
        
        print(f"Successfully copied {copied_count} images")
        if failed_count > 0:
            print(f"Failed to copy {failed_count} images")
        
        return copied_count
    
    def print_statistics(self, filtered_count: int, sampled_count: int, 
                        num_categories: int):
        """Print filtering statistics."""
        total_images = len(self.coco_data['images'])
        percentage = (filtered_count / total_images * 100) if total_images > 0 else 0
        
        print(f"{'='*60}")
        print(f"COCO Image Filtering Results")
        print(f"{'='*60}")
        print(f"Total images in dataset: {total_images:,}")
        print(f"Images with exactly {num_categories} unique category(ies): {filtered_count:,}")
        print(f"Percentage: {percentage:.2f}%")
        print(f"Sampled images: {sampled_count:,}")
        print(f"{'='*60}")


def main():
    """Main execution function."""
    
    # Configuration
    ANNOTATION_FILE_PATH = os.path.join(MSCOCO_DIR,'annotations/annotations/instances_train2014.json')
    IMAGES_SOURCE_DIR = os.path.join(MSCOCO_DIR,'mscoco/train2014')  # Source directory for images
    NUM_CATEGORIES = 1  # Number of unique categories to filter for
    MAX_SAMPLES = 1000  # Maximum number of images to sample (None for all)
    OUTPUT_FILE = f'filtered_{NUM_CATEGORIES}_category_images.json'
    OUTPUT_IMAGES_DIR = f'filtered_{NUM_CATEGORIES}_category_images/'  # Directory to copy images
    RANDOM_SEED = 42  # For reproducibility (None for random)
    COPY_IMAGES = True  # Whether to copy images to OUTPUT_IMAGES_DIR
    
    # Initialize filter
    print(f"Loading annotations from: {ANNOTATION_FILE_PATH}")
    filter_obj = COCOImageFilter(ANNOTATION_FILE_PATH)

    # Filter images
    print(f"\nFiltering images with exactly {NUM_CATEGORIES} unique category(ies)...")
    filtered_image_ids, unique_categories_found = filter_obj.filter_by_category_count(NUM_CATEGORIES)
    
    # Sample images if needed
    if MAX_SAMPLES is not None and len(filtered_image_ids) > MAX_SAMPLES:
        print(f"Sampling {MAX_SAMPLES} images from {len(filtered_image_ids)} filtered images...")
        sampled_image_ids = filter_obj.sample_images(
            filtered_image_ids, 
            MAX_SAMPLES,
            unique_categories_found,
            RANDOM_SEED
        )
    else:
        sampled_image_ids = filtered_image_ids
    
    # Get detailed image information
    image_info = filter_obj.get_image_info(sampled_image_ids)
    
    # Save results
    filter_obj.save_results(
        image_info, 
        OUTPUT_FILE,
        len(filtered_image_ids),
        NUM_CATEGORIES
    )
    
    # Print statistics
    filter_obj.print_statistics(
        len(filtered_image_ids),
        len(sampled_image_ids),
        NUM_CATEGORIES
    )
    
    print(f"\nResults saved to: {OUTPUT_FILE}")
    
    # Copy images to folder
    if COPY_IMAGES:
        copied = filter_obj.copy_images_to_folder(
            image_info,
            IMAGES_SOURCE_DIR,
            OUTPUT_IMAGES_DIR
        )
        print(f"\nImages copied to: {OUTPUT_IMAGES_DIR}")
    
    # Show first 10 examples
    if image_info:
        print(f"\nFirst {min(10, len(image_info))} examples:")
        for i, img in enumerate(image_info[:10], 1):
            print(f"{i}. {img['file_name']}")
            print(f"   Image ID: {img['image_id']}, Category IDs: {img['category_ids']}")


if __name__ == "__main__":
    main()

Loading annotations from: /netscratch/elmansoury/mscoco/train2014/annotations/annotations/instances_train2014.json

Filtering images with exactly 1 unique category(ies)...

Number of images per category in filtered results:
  Category ID 58: 81 images
  Category ID 18: 258 images
  Category ID 64: 20 images
  Category ID 72: 36 images
  Category ID 16: 898 images
  Category ID 17: 311 images
  Category ID 19: 460 images
  Category ID 20: 616 images
  Category ID 21: 665 images
  Category ID 44: 21 images
  Category ID 63: 7 images
  Category ID 62: 15 images
  Category ID 2: 43 images
  Category ID 3: 97 images
  Category ID 4: 206 images
  Category ID 5: 1098 images
  Category ID 6: 187 images
  Category ID 7: 886 images
  Category ID 9: 352 images
  Category ID 1: 238 images
  Category ID 13: 375 images
  Category ID 28: 73 images
  Category ID 32: 35 images
  Category ID 37: 4 images
  Category ID 54: 87 images
  Category ID 65: 232 images
  Category ID 77: 72 images
  Category ID 8

In [ ]:
# what are the categories that the sample images have ? we want to have all the categories balanced in the sample images.
